In [2]:
# Data handling
import pandas as pd
import numpy as np

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# NLP
from sklearn.feature_extraction.text import TfidfVectorizer

# Evaluation
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

In [3]:
# Load dataset

df = pd.read_csv("D:\Review risk\Womens Clothing E-Commerce Reviews.csv")

In [4]:
# Display first 5 rows

df.head()

,userid,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,0,33,NaN,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates
1,1,34,NaN,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses
2,2,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses
3,3,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants
4,4,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses


In [5]:
# Check missing values

df.isnull().sum()

userid                        0
Age                           0
Title                      3810
Review Text                 845
Rating                        0
Recommended IND               0
Positive Feedback Count       0
Division Name                14
Department Name              14
Class Name                   14
dtype: int64

In [6]:
# Remove rows with missing review text

df = df.dropna(subset=['Review Text'])

In [7]:
# Replace missing titles with empty string

df['Title'] = df['Title'].fillna('')

In [8]:
# Keywords indicating possible returns

keywords = [
    'small',
    'large',
    'tight',
    'loose',
    'poor quality',
    'cheap',
    'defective',
    'broken',
    'bad',
    'return'
]
# Create return labels

def create_return_label(row):

    review = str(row['Review Text']).lower()

    if row['Rating'] <= 2:
        return 1

    for word in keywords:

        if word in review:
            return 1

    return 0
    

In [9]:
# Create target column

df['Returned'] = df.apply(
    create_return_label,
    axis=1
)

In [10]:
# Count return labels

df['Returned'].value_counts()

Returned
0    12073
1    10568
Name: count, dtype: int64

TF-IDF

🔥 THIS IS THE NEW PART

Instead of creating Complaint Category.

Use Review Text directly.

In [11]:
# Convert text into numerical vectors

tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=5000
)

In [12]:
# Transform review text into TF-IDF matrix

X = tfidf.fit_transform(
    df['Review Text']
)
# Target variable

y = df['Returned']

In [13]:
# Check dimensions

print(X.shape)
print(y.shape)

(22641, 5000)
(22641,)


In [14]:
# Split data

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [15]:
# Create model

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

In [16]:
# Train model

rf_model.fit(X_train,y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [17]:
# Predict labels

y_pred = rf_model.predict(
    X_test
)

In [18]:
# Calculate accuracy

accuracy = accuracy_score(
    y_test,
    y_pred
)

print("Accuracy:", accuracy)

Accuracy: 0.9383969971296092


In [19]:
# Detailed evaluation

print(
    classification_report(
        y_test,
        y_pred
    )
)

              precision    recall  f1-score   support

           0       0.93      0.96      0.94      2472
           1       0.95      0.91      0.93      2057

    accuracy                           0.94      4529
   macro avg       0.94      0.94      0.94      4529
weighted avg       0.94      0.94      0.94      4529



In [20]:
# Probability prediction

y_prob = rf_model.predict_proba(
    X_test
)

print(y_prob[:5])

[[0.35 0.65]
 [0.17 0.83]
 [0.07 0.93]
 [0.37 0.63]
 [0.52 0.48]]


In [21]:
# Example review

new_review = [
    "Beautiful dress but size runs small and quality is poor"
]
# Convert review into TF-IDF format

review_vector = tfidf.transform(
    new_review
)

In [22]:
# Get probability

risk = rf_model.predict_proba(
    review_vector
)[0][1] * 100

print(f"Return Risk: {risk:.2f}%")

Return Risk: 90.00%


In [23]:
# Customer review
review = [
    "The dress looks beautiful but the size runs very small. The fabric quality is poor and I am disappointed with the purchase."
]

# Convert review into TF-IDF vector
review_vector = tfidf.transform(review)

# Predict return probability
risk = rf_model.predict_proba(review_vector)[0][1] * 100

# Display result
print(f"Return Risk: {risk:.2f}%")

Return Risk: 88.00%


In [24]:
# Customer review
review = [
    "The dress fits perfectly. Excellent quality, beautiful color and I absolutely love it. Highly recommended."
]

# Convert review into TF-IDF vector
review_vector = tfidf.transform(review)

# Predict return probability
risk = rf_model.predict_proba(review_vector)[0][1] * 100

# Display result
print(f"Return Risk: {risk:.2f}%")

Return Risk: 5.00%


In [25]:
# Customer review
review = [
    "The dress is good and the quality is decent, but the color is slightly different from the picture."
]

# Convert review into TF-IDF vector
review_vector = tfidf.transform(review)

# Predict return probability
risk = rf_model.predict_proba(review_vector)[0][1] * 100

# Display result
print(f"Return Risk: {risk:.2f}%")

Return Risk: 16.00%
